# N6 — Experiments, ablations and bootstrap

In [ ]:
from pathlib import Path
import sys, yaml, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
CFG=yaml.safe_load((ROOT/'config.yaml').read_text())
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('device:',DEVICE,'debug:',CFG['data']['debug'])

from piano_transcriber.metrics import frame_metrics,paired_bootstrap

Protocol: evaluate all methods on identical held-out clips. Tune lambda and thresholds on validation only. Save one row per clip for paired bootstrap.

In [ ]:
cache=ROOT/'data'/'cached_test_predictions.npz'
if cache.exists():
 z=np.load(cache);probs,targets=z['probs'],z['targets'];print(probs.shape,targets.shape)
else:print('Export aligned test probabilities and labels from N2 before running final comparisons.')

In [ ]:
lambdas=[0,.1,.25,.5,.75,1.]
display(pd.DataFrame([{'lambda':x,'frame_f1':np.nan,'note_f1':np.nan} for x in lambdas]))
ablation=pd.DataFrame([{'factor':'CD-k','level':k} for k in [1,3,5]]+
 [{'factor':'hidden units','level':h} for h in [128,256,512]]+
 [{'factor':'previous-frame context','level':v} for v in [False,True]]+
 [{'factor':'noise SNR dB','level':v} for v in [None,30,20,10]])
display(ablation)

Run each ablation with all other settings fixed. Report mean and standard deviation across seeds where feasible. Use paired bootstrap on per-clip differences; confidence intervals crossing zero do not support a clear improvement.